# Exploratory Data Analysis

## Initial Exploration

With the objective of understanding the data structure regarding the STAR-TPM Glioblastoma gene expression dataset, this present stage of the investigation will follow the standard *Exploratory Data Analysis* methodology for uncovering the contents of the dataset.

The following code will help us understand:

- Number of samples (patients)
- Number of variables (genes)
- Available clinical variables

In [11]:
import pandas as pd

df = pd.read_csv('../../data/star_data/rnaseq_stpm_transposed.csv')

# 1. Count the number of samples
num_samples = len(df)

# 2. Count the total number of variables
num_variables = len(df.columns)

# 3. Identify and count available clinical variables
# Note: In UCSC Xena datasets, gene expression columns are usually numeric, 
# while clinical variables are often non-numeric or specifically labeled.
clinical_vars = df.select_dtypes(exclude=['float64', 'int64']).columns.tolist()
num_clinical_vars = len(clinical_vars)

print(f"Number of samples: {num_samples}")
print(f"Total number of variables: {num_variables}")
print(f"Number of clinical variables: {num_clinical_vars}")
print(f"Clinical variables found: {clinical_vars}")


Number of samples: 175
Total number of variables: 60661
Number of clinical variables: 1
Clinical variables found: ['Patient_Barcode']


As we can see, there are n=175 patient  samples in the dataset, with p=60,660 variables, each of which correspond to a specific RNA sequence gene expressions. The only clinical variable that was found in the dataset is `Ensembl_ID`, which pertains to the unique identifier for each of the n gene expressions, and as such, will not be included in the extent of this project.

According to the National Cancer Institute pertaining to the U.S. Department of Health and Human Services, the creation of this unique sequence expression was derived from the [mRNA Analysis Pipeline](https://docs.gdc.cancer.gov/Data/Bioinformatics_Pipelines/Expression_mRNA_Pipeline/), which is a quantification analysis pipeline that measures gene level expression with the [STAR](https://github.com/alexdobin/STAR/blob/master/doc/STARmanual.pdf) aligner code, which uses TPM (Transcripts Per Million) values to map RNA sequence reads. This method is creates reliable and normalized measures for gene expressions, which are optimal for visualization via unsupervised learning methods accross samples. 

Furthermore, it is worth stating that the present dataset, and the one which will be used in the rest of the research investigation is exactly the same as the original [GDC TCGA Glioblastoma Dataset](https://xenabrowser.net/datapages/?cohort=GDC%20TCGA%20Glioblastoma%20(GBM)&removeHub=https%3A%2F%2Fxena.treehouse.gi.ucsc.edu%3A443), with the difference in this dataset being that the present has been transposed so as to align the data with bioinformatics and machine learning standards. Most python libraries focused on machine learning expect columns and rows to represent samples and features/genes respectively, and *TCGA GBM* datasets structure their datasets the other way around.

### Missing data, outliers and general data distribution

With the objective of further understanding the present dataset, the following stage of the *exploratory analysis* phase of the project will be centered around finding the general distribution of the data, as well as finding missing data and outliers. 

In accordance to level of difficulty, we now proceed to check for missing data in the dataframe.

In [12]:
import numpy as np

def analyze_missing_data(df):
    """
    Counts missing data and calculates the percentage for each column.
    Returns a DataFrame sorted by the highest percentage of missing values.
    """
    missing_counts = df.isnull().sum()
    missing_percentages = (df.isnull().sum() / len(df)) * 100
    
    missing_summary = pd.DataFrame({
        'Missing Values': missing_counts,
        'Percentage (%)': missing_percentages
    })
    
    # Filter to show only columns that actually have missing data
    missing_summary = missing_summary[missing_summary['Missing Values'] > 0]
    
    # Sort by the highest percentage of missing data
    missing_summary = missing_summary.sort_values(by='Percentage (%)', ascending=False)
    
    return missing_summary

# --- Example Usage ---
print("--- Missing Data Summary ---")
print(analyze_missing_data(df))


--- Missing Data Summary ---
Empty DataFrame
Columns: [Missing Values, Percentage (%)]
Index: []


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import skew


def process_gene_data(df):
    """
    Cleans data, filters for high-variance genes, and plots distribution statistics.
    """
    # 1. Handle Missing Values
    # We'll use median imputation to stay robust against outliers
    #df_cleaned = df.fillna(df.median())

    # 2. Handle Potential Outliers
    # Using the IQR method to clip extreme values (winsorization) 
    # This keeps the data points but prevents them from distorting the variance too much.
    #Q1 = df_cleaned.quantile(0.25)
    #Q3 = df_cleaned.quantile(0.75)
    #IQR = Q3 - Q1
    #lower_bound = Q1 - 1.5 * IQR
    #upper_bound = Q3 + 1.5 * IQR
    #df_cleaned = df_cleaned.clip(lower=lower_bound, upper=upper_bound, axis=1)

    # 3. Calculate Top K Variance Filter (Top 15)
    variances = df.var().sort_values(ascending=False)
    top_15_genes = variances.head(15).index
    df_top_15 = df[top_15_genes]
    
    # 4. Violin Plot for Top 15 Most Variable Genes
    plt.figure(figsize=(15, 6))
    # Melting the dataframe makes it compatible with Seaborn's categorical plotting
    df_melted = df_top_15.melt(var_name='Gene', value_name='Expression')
    sns.violinplot(data=df_melted, x='Gene', y='Expression', hue='Gene', palette='viridis', legend=False)
    plt.xticks(rotation=45)
    plt.title('Distribution of Top 15 Most Variable Genes')
    plt.tight_layout()
    plt.show()

    # 5. Calculate Global Statistics for all 175 genes
    stats_df = pd.DataFrame({
        'Mean': df.mean(),
        'Variance': df.var(),
        'Skewness': df.apply(lambda x: skew(x))
    })

    # 6. Plot Histograms for Mean, Variance, and Skewness
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Mean Histogram
    sns.histplot(stats_df['Mean'], kde=True, ax=axes[0], color='skyblue')
    axes[0].set_title('Distribution of Gene Means')

    # Variance Histogram
    sns.histplot(stats_df['Variance'], kde=True, ax=axes[1], color='salmon')
    axes[1].set_title('Distribution of Gene Variances')

    # Skewness Histogram
    sns.histplot(stats_df['Skewness'], kde=True, ax=axes[2], color='lightgreen')
    axes[2].set_title('Distribution of Gene Skewness')

    plt.tight_layout()
    plt.show()

    return stats_df

# Execute the analysis
stats_summary = process_gene_data(df)


TypeError: could not convert string to float: 'TCGA-27-2521-01A'

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
